## **User Viewing Behavior Analysis in stc TV**

### **Data Analysis for STC TV Dataset**

**Read data**

In [ ]:
#import necessary libraries
!pip install pyxlsb
import pandas as pd
from google.colab import drive

#Mount Google Drive
drive.mount('/content/drive')

#Load Dataset
df = pd.read_excel('/content/drive/MyDrive/stc TV Data Set_T1.xlsb', engine='pyxlsb')

#Display First 5 rows
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Column1,date_,user_id_maped,program_name,duration_seconds,program_class,season,episode,program_desc,program_genre,series_title,hd,original_name
0,1,42882,26138,100 treets,40,MOVIE,0,0,Drama Movie100 Streets,Drama,0,0,100 treets
1,3,42876,7946,Moana,17,MOVIE,0,0,Animation MovieMoana (HD),Animation,0,1,Moana
2,4,42957,7418,The Mermaid Princess,8,MOVIE,0,0,Animation MovieThe Mermaid Princess (HD),Animation,0,1,The Mermaid Princess
3,5,42942,19307,The Mermaid Princess,76,MOVIE,0,0,Animation MovieThe Mermaid Princess (HD),Animation,0,1,The Mermaid Princess
4,7,42923,15860,Churchill,87,MOVIE,0,0,Biography MovieChurchill (HD),Biography,0,1,Churchill


**Data Exploration and Structure Check**

In [ ]:
#Check Dataset Shape
df.shape

#check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 13 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   Column1           1048575 non-null  int64 
 1   date_             1048575 non-null  int64 
 2   user_id_maped     1048575 non-null  int64 
 3   program_name      1048575 non-null  object
 4   duration_seconds  1048575 non-null  int64 
 5   program_class     1048575 non-null  object
 6   season            1048575 non-null  int64 
 7   episode           1048575 non-null  int64 
 8   program_desc      1034537 non-null  object
 9   program_genre     1048575 non-null  object
 10  series_title      1048575 non-null  int64 
 11  hd                1048575 non-null  int64 
 12  original_name     1048575 non-null  object
dtypes: int64(8), object(5)
memory usage: 104.0+ MB


**Preprocessing**

In [ ]:
# Data Preprocessing on the input data

# 1) Drop index column if exists
if 'Column1' in df.columns:
    df = df.drop(columns=['Column1'])

# 2) Trim spaces in program names
df['program_name'] = df['program_name'].astype(str).str.strip()

# 3) Convert date_ to datetime
df['date_'] = pd.to_datetime(df['date_'], errors='coerce')

# 4) Ensure numeric types
num_cols = ['duration_seconds','season','episode','hd']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df.head()

,date_,user_id_maped,program_name,duration_seconds,program_class,season,episode,program_desc,program_genre,series_title,hd,original_name
0,1970-01-01 00:00:00.000042882,26138,100 treets,40,MOVIE,0,0,Drama Movie100 Streets,Drama,0,0,100 treets
1,1970-01-01 00:00:00.000042876,7946,Moana,17,MOVIE,0,0,Animation MovieMoana (HD),Animation,0,1,Moana
2,1970-01-01 00:00:00.000042957,7418,The Mermaid Princess,8,MOVIE,0,0,Animation MovieThe Mermaid Princess (HD),Animation,0,1,The Mermaid Princess
3,1970-01-01 00:00:00.000042942,19307,The Mermaid Princess,76,MOVIE,0,0,Animation MovieThe Mermaid Princess (HD),Animation,0,1,The Mermaid Princess
4,1970-01-01 00:00:00.000042923,15860,Churchill,87,MOVIE,0,0,Biography MovieChurchill (HD),Biography,0,1,Churchill


In [ ]:
#Summary statistics for numerical columns
df.describe()

,date_,user_id_maped,duration_seconds,season,episode,series_title,hd
count,1048575,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06
mean,1970-01-01 00:00:00.000043012,1.709266e+04,1.230957e+03,1.342139e+00,6.157952e+00,1.205922e-02,3.862728e-01
min,1970-01-01 00:00:00.000042808,1.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1970-01-01 00:00:00.000042896,8.253000e+03,5.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1970-01-01 00:00:00.000043022,1.714900e+04,1.190000e+02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
75%,1970-01-01 00:00:00.000043121,2.566500e+04,1.328000e+03,1.000000e+00,9.000000e+00,0.000000e+00,1.000000e+00
max,1970-01-01 00:00:00.000043220,3.428000e+04,1.461329e+06,2.300000e+01,2.820000e+02,1.000000e+00,1.000000e+00
std,NaN,1.003513e+04,6.821058e+03,2.104095e+00,1.222015e+01,1.091504e-01,4.868946e-01


The dataset contains over one million viewing records with 13 variables describing user behavior and program characteristics. The data is mostly clean, with missing values only in the program description column, which does not affect behavioral analysis.

In [ ]:
#check missing values
df.isnull().any()

,0
date_,False
user_id_maped,False
program_name,False
duration_seconds,False
program_class,False
season,False
episode,False
program_desc,True
program_genre,False
series_title,False


# **Task 1**

In [ ]:
# Task 1: Most watched movies/series (Total Views / Total Users / Total watch time)

df_task1 = df.copy()

# For series/episodes: append season and episode to differentiate
mask = df_task1['program_class'].astype(str).str.upper().eq('SERIES/EPISODES')
df_task1.loc[mask, 'program_name'] = (
    df_task1.loc[mask, 'program_name'].astype(str)
    + '_SE' + df_task1.loc[mask, 'season'].astype('Int64').astype(str)
    + '_EP' + df_task1.loc[mask, 'episode'].astype('Int64').astype(str)
)

grouped_programs = (
    df_task1.groupby(['program_name','program_class'])
    .agg({'user_id_maped':[('c01','nunique'), ('c02','count')],
          'duration_seconds':[('c03','sum')]})
    .reset_index()
)

grouped_programs.columns = ['program_name','program_class','No of Users who Watched','No of watches','Total watch time in seconds']
grouped_programs['Total watch time in hours'] = grouped_programs['Total watch time in seconds'] / 3600
grouped_programs = grouped_programs.drop(columns=['Total watch time in seconds'])

grouped_programs = grouped_programs.sort_values(
    by=['Total watch time in hours','No of watches','No of Users who Watched'],
    ascending=False
).reset_index(drop=True)

# show top 10
grouped_programs.head(10)

,program_name,program_class,No of Users who Watched,No of watches,Total watch time in hours
0,The Boss Baby,MOVIE,3389,24047,2961.350833
1,The Amazing pider-Man,MOVIE,1011,2877,1966.119167
2,The Expendables,MOVIE,853,2119,1961.159444
3,Moana,MOVIE,2173,8081,1706.176944
4,Trolls,MOVIE,2613,13793,1601.023056
5,Bean,MOVIE,949,3617,1423.955000
6,The murfs,MOVIE,867,3132,1342.141111
7,Hotel Transylvania,MOVIE,491,1947,1096.533611
8,Cloudy With a Chance of Meatballs,MOVIE,683,2076,948.674722
9,The Man With The Iron Fists,MOVIE,707,2505,859.626389


**Top 10 Most Watched Programs**

In [ ]:
# Visualization libraries
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pie chart for top 10 programs by watch time (hours)
fig_top10 = px.pie(
    grouped_programs.head(10),
    values='Total watch time in hours',
    names='program_name',
    hover_data=['program_class'],
    title='Top 10 programs in total watch time (hours)'
)
fig_top10.update_traces(sort=False)
fig_top10.show()

**Top Watched Programs Insight**

The analysis shows that several popular movies dominate the platform in terms of total watch time.
For example, "The Boss Baby" has the highest total viewing time among all programs.
This indicates that family and animated movies attract a large audience and generate significant engagement on the platform.

**Viewing Behavior by program Class**

In [ ]:
# Study customer experience against Program class

grouped_class = df.copy()
grouped_class = (
    grouped_class.groupby('program_class')
    .agg({'user_id_maped':[('c01','nunique'), ('c02','count')],
          'duration_seconds':[('c03','sum')]})
    .reset_index()
)

grouped_class.columns = ['program_class','No of Users who Watched','No of watches','Total watch time in seconds']
grouped_class['Total watch time in hours'] = grouped_class['Total watch time in seconds'] / 3600
grouped_class = grouped_class.drop(columns=['Total watch time in seconds'])

grouped_class = grouped_class.sort_values(
    by=['Total watch time in hours','No of watches','No of Users who Watched'],
    ascending=False
).reset_index(drop=True)

grouped_class.head(35)

,program_class,No of Users who Watched,No of watches,Total watch time in hours
0,SERIES/EPISODES,3901,560174,255097.787500
1,MOVIE,11355,488401,103444.145556


In [ ]:
fig1 = px.pie(
    grouped_class,
    values='Total watch time in hours',
    names='program_class',
    hover_data=['program_class'],
    title='Total duration spent by program_class'
)

fig2 = px.pie(
    grouped_class,
    values='No of Users who Watched',
    names='program_class',
    hover_data=['program_class'],
    title='Total Users watching by program_class'
)

fig1.update_traces(sort=False)
fig2.update_traces(sort=False)

fig1.show()
fig2.show()

**Program Class Analysis**

The results show that series and episodic content account for the majority of total watch time compared to movies.
Although movies have a large number of viewers, users tend to spend more time watching series because they consist of multiple episodes.

**HD vs SD Viewing Behavior**

In [ ]:
# Study the relation and user's behaviour against HD flag

grouped_hd = df.copy()
grouped_hd = (
    grouped_hd.groupby('hd')
    .agg({'user_id_maped':[('c01','nunique'), ('c02','count')],
          'duration_seconds':[('c03','sum')]})
    .reset_index()
)

grouped_hd.columns = ['hd','No of Users who Watched','No of watches','Total watch time in seconds']
grouped_hd['Total watch time in hours'] = grouped_hd['Total watch time in seconds'] / 3600
grouped_hd = grouped_hd.drop(columns=['Total watch time in seconds'])

grouped_hd = grouped_hd.sort_values(
    by=['Total watch time in hours','No of watches','No of Users who Watched'],
    ascending=False
).reset_index(drop=True)

grouped_hd

,hd,No of Users who Watched,No of watches,Total watch time in hours
0,0,6728,643539,268364.372778
1,1,11000,405036,90177.560278


In [ ]:
# Optional: map 0/1 to SD/HD for readability
grouped_hd_display = grouped_hd.copy()
grouped_hd_display['Quality'] = grouped_hd_display['hd'].map({0:'SD', 1:'HD'}).fillna(grouped_hd_display['hd'].astype(str))

fig_hd_time = px.pie(
    grouped_hd_display,
    values='Total watch time in hours',
    names='Quality',
    title='Total watch time (hours): HD vs SD'
)

fig_hd_users = px.pie(
    grouped_hd_display,
    values='No of Users who Watched',
    names='Quality',
    title='Total users: HD vs SD'
)

fig_hd_time.update_traces(sort=False)
fig_hd_users.update_traces(sort=False)

fig_hd_time.show()
fig_hd_users.show()

**HD vs SD Viewing Behavior**

The analysis shows that most viewing time is spent on SD content, while HD viewing represents a smaller portion.
However, a large number of users still watch in HD quality, indicating that users value higher video quality when available.

# **Conclusion**

This analysis explored user viewing behavior on the stc TV platform using more than one million viewing records.
The results show that animated and popular movies receive the highest watch time, while series dominate overall viewing duration due to multiple episodes.
Additionally, although most viewing occurs in SD quality, HD content still attracts a considerable number of users.
These insights help understand user preferences and can support content recommendation and platform optimization.